In [ ]:
%load_ext autoreload
%autoreload 2

# 1. DQN Agent

In [ ]:
import numpy as np
import torch

from briscola import Briscola
from agents.dqn_agent import DQN_Agent, state_to_tensor

In [ ]:
N_EPISODES = 10_000
LOG_INTERVAL = 500

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [ ]:
env = Briscola()
try:
    model = "models/150000_dqn.pt"
    save = torch.load(model, map_location = device)
    agent = DQN_Agent(env, device, save)
    print(f"Agent loaded from checkpoint: {model}")
except:
    agent = DQN_Agent(env, device)
    print("Agent created from scratch.")

In [ ]:
episode_rewards = [] # Hopefully higher as episodes increase
wins = []
agent.epsilon = 0
agent.eps_end = 0
agent.policy_net.eval()

for i in range(N_EPISODES):
    done = False
    ep_reward = 0.0
    
    state, _ = env.reset()
    
    while not done:
        action = agent.select_action(state).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        next_mask = next_state["hand"]
        done = terminated or truncated

        ep_reward += reward
        
        # Convert to tensors to store in experience buffer
        s_tensor = state_to_tensor(state).to(agent.device)
        r_tensor = torch.tensor([reward], dtype = torch.float32, device = agent.device)
        a_tensor = torch.tensor([[action]], device = agent.device, dtype = torch.long)

        if terminated:
            ns_tensor = None
            nm_tensor = None
        else:
            ns_tensor = state_to_tensor(next_state).to(agent.device)
            nm_tensor = torch.tensor(next_mask, dtype = torch.float32, device = agent.device).unsqueeze(0)
        
        # Buffer update
        agent.buffer.push(s_tensor, a_tensor, ns_tensor, r_tensor, nm_tensor)
        
        state = next_state
        action_mask = next_mask
        
        # Execute a learning step
        agent.learn()

    # agent.epsilon = max(agent.eps_end, initial_epsilon - (i / N_EPISODES) * (initial_epsilon - agent.eps_end))
    
    episode_rewards.append(ep_reward)
    if env.player_score > 60:
        wins.append(1)
    else:
        wins.append(0)

    if (i + 1) % LOG_INTERVAL == 0:
        recent      = episode_rewards[-LOG_INTERVAL:]
        win_rate    = np.mean(wins) * 100
        recent_wins = np.mean(wins[-LOG_INTERVAL:]) * 100
        print(
            f"Episode {i+1:>6}/{N_EPISODES} | "
            f"Mean reward (last {LOG_INTERVAL}): {np.mean(recent):+.2f} | "
            f"Win rate: {win_rate:.1f}% | "
            f"Win rate (last {LOG_INTERVAL}): {recent_wins:.1f}% | "
            f"Epsilon: {agent.epsilon:.4f}"
        )

torch.save(agent.policy_net.state_dict(), "models/NAMEHOLDER_dqn.pt")
print("Saved.")

In [ ]:
env   = Briscola()
save = torch.load("models/150000_dqn.pt", map_location = device)
agent = DQN_Agent(
    env=env,
    device=device,
    savefile=save,
    epsilon_start=0.0,   # no exploration during eval
    epsilon_end=0.0,
)
agent.policy_net.eval()

# ── Trackers ─────────────────────────────────────────────────
results        = []   # "win" / "loss" / "tie"
score_diffs    = []   # player_score - opponent_score
episode_rewards = []
points_in_outcome = {"win": [], "loss": [], "tie": []}

for ep in range(N_EPISODES):
    state, info = env.reset()
    ep_reward   = 0.0
    done        = False

    while not done:
        action = agent.select_action(state).item()
        state, reward, terminated, truncated, info = env.step(action)
        ep_reward += reward
        done = terminated or truncated

    episode_rewards.append(ep_reward)

    if env.player_score > 60:
        outcome = "win"
    elif env.player_score < 60:
        outcome = "loss"
    else:
        outcome = "tie"

    results.append(outcome)
    points_in_outcome[outcome].append(env.player_score)

# ── Report ────────────────────────────────────────────────────
wins   = results.count("win")
losses = results.count("loss")
ties   = results.count("tie")

print(f"\n{'='*52}")
print(f"  Evaluation over {N_EPISODES} episodes")
print(f"{'='*52}")
print(f"  Win rate:   {wins/N_EPISODES*100:5.1f}%  ({wins})")
print(f"  Loss rate:  {losses/N_EPISODES*100:5.1f}%  ({losses})")
print(f"  Tie rate:   {ties/N_EPISODES*100:5.1f}%  ({ties})")
print(f"{'─'*52}")
print(f"  Avg shaped reward per episode:      {np.mean(episode_rewards):+.2f}")
print(f"{'─'*52}")
for outcome in ("win", "loss", "tie"):
    pts = points_in_outcome[outcome]
    if pts:
        print(f"  Avg agent points in {outcome}s:         {np.mean(pts):.1f}")
print(f"{'='*52}\n")

In [ ]:
# set up matplotlib
# is_ipython = 'inline' in matplotlib.get_backend()
# if is_ipython:
#     from IPython import display

# plt.ion()

In [ ]:
# def plot_rewards(show_result=False):
#     plt.figure(1)
#     rewards_t = torch.tensor(episode_rewards, dtype=torch.float)

#     if show_result:
#         plt.title('Result')
#     else:
#         plt.clf()
#         plt.title('Training...')

#     plt.xlabel('Episode')
#     plt.ylabel('Total Reward')

#     # Plot raw rewards
#     # plt.plot(rewards_t.numpy(), label="Episode Reward")

#     # Plot moving average (100 episodes)
#     # if len(rewards_t) >= 100:
#     if len(rewards_t) >= 10:
#         # means = rewards_t.unfold(0, 100, 1).mean(1).view(-1)
#         means = rewards_t.unfold(0, 10, 1).mean(1).view(-1)
#         # means = torch.cat((torch.zeros(99), means))
#         means = torch.cat((torch.zeros(9), means))
#         plt.plot(means.numpy(), label="10-episode avg")

#     plt.legend()
#     # plt.pause(0.001)

#     if is_ipython:
#         if not show_result:
#             display.display(plt.gcf())
#             display.clear_output(wait=True)
#         else:
#             display.display(plt.gcf())

# 2. PPO Agent

In [ ]:
import numpy as np
import torch
from briscola import Briscola
from agents.ppo_agent import BriscolaNet, PPO, RolloutBuffer

# ── Hyperparameters ──────────────────────────────────────────
ROLLOUT_STEPS  = 2048   # steps to collect before each update
TOTAL_STEPS    = 1_000_000
LR             = 3e-4
GAMMA          = 0.99
GAE_LAMBDA     = 0.95
CLIP_EPS       = 0.2
VALUE_COEF     = 0.5
ENTROPY_COEF   = 0.03   # increase to 0.05 if agent gets stuck early
N_EPOCHS       = 10
BATCH_SIZE     = 64
MAX_GRAD_NORM  = 0.5
HIDDEN         = 128
LOG_INTERVAL   = 10     # log every N updates
# ─────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu")

def flatten_obs(obs: dict) -> np.ndarray:
    return np.concatenate([
        obs["hand"].astype(np.float32),
        obs["table_card"].astype(np.float32),
        obs["briscola"].astype(np.float32),
        obs["played_cards"].astype(np.float32),
        obs["is_first"].astype(np.float32),
    ])

env       = Briscola()
obs_dim   = len(flatten_obs(env.reset()[0]))
n_actions = env.action_space.n   # 40

net = BriscolaNet(obs_dim, n_actions, hidden=HIDDEN).to(device)
agent = PPO(
    net, lr=LR, gamma=GAMMA, gae_lambda=GAE_LAMBDA,
    clip_eps=CLIP_EPS, value_coef=VALUE_COEF,
    entropy_coef=ENTROPY_COEF, n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE, max_grad_norm=MAX_GRAD_NORM,
    device=str(device),
)

buffer = RolloutBuffer()

obs, info   = env.reset()
state       = flatten_obs(obs)
action_mask = info["action_masks"]

episode_rewards = []
ep_reward       = 0.0
total_steps     = 0
n_updates       = 0

while total_steps < TOTAL_STEPS:

    # ── Phase 1: collect ROLLOUT_STEPS transitions ──
    buffer.clear()
    for _ in range(ROLLOUT_STEPS):
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        mask_t  = torch.tensor(action_mask, dtype=torch.bool, device=device).unsqueeze(0)

        with torch.no_grad():
            action, log_prob, _, value = net.get_action(state_t, mask_t)

        a = action.item()
        next_obs, reward, terminated, truncated, info = env.step(a)
        done = terminated or truncated

        buffer.states.append(state)
        buffer.actions.append(a)
        buffer.log_probs.append(log_prob.item())
        buffer.rewards.append(reward)
        buffer.values.append(value.item())
        buffer.masks.append(action_mask)
        buffer.dones.append(float(done))

        ep_reward += reward
        total_steps += 1

        if done:
            episode_rewards.append(ep_reward)
            ep_reward = 0.0
            obs, info = env.reset()
        else:
            obs = next_obs

        state       = flatten_obs(obs)
        action_mask = info["action_masks"]

    # ── Phase 2 & 3: compute GAE + update ──
    last_state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    last_mask_t  = torch.tensor(action_mask, dtype=torch.bool, device=device).unsqueeze(0)
    with torch.no_grad():
        _, last_value = net(last_state_t)
    last_value = last_value.item()

    stats = agent.update(buffer, last_value)
    n_updates += 1

    if n_updates % LOG_INTERVAL == 0 and episode_rewards:
        recent = episode_rewards[-100:]
        print(
            f"Steps: {total_steps:>8,} | "
            f"Updates: {n_updates:>4} | "
            f"Mean reward (100 ep): {np.mean(recent):+.2f} | "
            f"Policy loss: {stats['policy_loss']:+.4f} | "
            f"Value loss: {stats['value_loss']:.4f} | "
            f"Entropy: {stats['entropy']:.4f}"
        )

torch.save(net.state_dict(), "briscola_ppo.pt")
print("Saved.")